# Otimização dos Modelos

Este notebook realiza a otimização dos modelos treinados sobre a base V2.

Os objetivos desta etapa são:

- manter a mesma estratégia de treino e validação utilizada anteriormente;
- testar diferentes configurações de hiperparâmetros;
- avaliar um modelo Gradient Boosted Trees;
- comparar os modelos pela capacidade discriminatória e pelo desempenho na classe de alunos não alfabetizados;
- selecionar o modelo final antes da avaliação temporal em 2024.

O conjunto de teste de 2024 permanecerá reservado durante toda esta etapa.

# 1. Configuração

Nesta etapa são importadas as bibliotecas utilizadas na otimização e avaliação dos modelos.

In [0]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier,
)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.ml.functions import vector_to_array

# 2. Carregamento da Base V2

É utilizada a segunda versão da base analítica, enriquecida com indicadores históricos do IDEB.

A mesma tabela de splits é reutilizada para preservar a comparabilidade com os experimentos anteriores.

In [0]:
df_gold = spark.table("workspace.gold.ml_alfabetizacao_aluno_v2")

df_split = spark.table("workspace.gold.ml_split_modelagem")

# 3. Preparação dos Dados

Os conjuntos de treino e validação são reconstruídos utilizando os mesmos municípios definidos anteriormente.

O conjunto de teste de 2024 não será utilizado durante a otimização.

In [0]:
df_modelagem = df_gold.join(
    df_split,
    on=["ano", "id_municipio"],
    how="inner",
)

df_treino = df_modelagem.filter(F.col("split") == "treino")

df_validacao = df_modelagem.filter(F.col("split") == "validacao")

In [0]:
display(
    df_modelagem.filter(F.col("split").isin("treino", "validacao"))
    .groupBy("split")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(
            F.avg(F.col("alfabetizado").cast("double")) * 100,
            2,
        ).alias("taxa_alfabetizacao"),
    )
    .orderBy("split")
)

# 4. Definição das Features

São utilizadas as mesmas features da segunda rodada de modelagem.

Dessa forma, as diferenças de desempenho desta etapa estarão associadas à configuração dos modelos, e não a novas alterações na base.

In [0]:
target = "alfabetizado"

features_categoricas = ["rede"]

features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
    "ideb_saeb_portugues_2021",
    "ideb_indicador_rendimento_2021",
]

features_modelo = features_categoricas + features_numericas

# 5. Pipeline de Preprocessing

O mesmo preprocessing da V2 é reutilizado.

Os parâmetros de imputação, codificação e padronização continuam sendo ajustados exclusivamente sobre o conjunto de treino.

In [0]:
df_treino_ml = df_treino.withColumn("label", F.col("alfabetizado").cast("double"))

df_validacao_ml = df_validacao.withColumn("label", F.col("alfabetizado").cast("double"))

features_numericas_imputadas = [f"{coluna}_imp" for coluna in features_numericas]

imputer = Imputer(
    inputCols=features_numericas,
    outputCols=features_numericas_imputadas,
    strategy="median",
)

rede_indexer = StringIndexer(
    inputCol="rede",
    outputCol="rede_index",
    handleInvalid="keep",
)

rede_encoder = OneHotEncoder(
    inputCols=["rede_index"],
    outputCols=["rede_ohe"],
    handleInvalid="keep",
)

assembler_tree = VectorAssembler(
    inputCols=(
        features_numericas_imputadas
        + ["rede_ohe"]
    ),
    outputCol="features",
)

assembler_logistic = VectorAssembler(
    inputCols=(
        features_numericas_imputadas
        + ["rede_ohe"]
    ),
    outputCol="features_raw",
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False,
)

# 6. Métricas de Avaliação

Os modelos serão avaliados principalmente pelo ROC-AUC.

Também serão acompanhados acurácia, precision, recall e F1-score, com atenção especial à classe `0`, correspondente aos alunos não alfabetizados.

In [0]:
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

In [0]:
def calcular_metricas_classificacao(df_pred):
    resumo = df_pred.agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("label") == 0) & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("label") == 1) & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    ).collect()[0]

    tp = resumo["tp"]
    tn = resumo["tn"]
    fp = resumo["fp"]
    fn = resumo["fn"]

    accuracy = (tp + tn) / resumo["total"]

    precision_1 = tp / (tp + fp) if (tp + fp) else 0

    recall_1 = tp / (tp + fn) if (tp + fn) else 0

    f1_1 = (
        2 * precision_1 * recall_1 / (precision_1 + recall_1)
        if (precision_1 + recall_1)
        else 0
    )

    precision_0 = tn / (tn + fn) if (tn + fn) else 0

    recall_0 = tn / (tn + fp) if (tn + fp) else 0

    f1_0 = (
        2 * precision_0 * recall_0 / (precision_0 + recall_0)
        if (precision_0 + recall_0)
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_0,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_1,
    }

# 7. Referência de Desempenho

A Random Forest da V2 apresentou o melhor resultado até o momento, com ROC-AUC de **0,6130**.

Esse valor será utilizado como referência durante a otimização.

In [0]:
auc_referencia = 0.6130

print(f"ROC-AUC de referência: {auc_referencia:.4f}")

# 8. Otimização da Logistic Regression

A Logistic Regression apresentou ROC-AUC de **0,6064** na V2.

Nesta etapa são avaliadas diferentes intensidades e tipos de regularização, mantendo o restante do pipeline inalterado.

A seleção será realizada exclusivamente sobre a base de validação de 2023.

## 8.1 Preprocessing

In [0]:
pipeline_preprocessamento_logistic = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler_logistic,
        scaler,
    ]
)

modelo_preprocessamento_logistic = (
    pipeline_preprocessamento_logistic
    .fit(df_treino_ml)
)

In [0]:
df_treino_logistic = (
    modelo_preprocessamento_logistic.transform(df_treino_ml)
    .select("label", "features")
    .cache()
)

df_validacao_logistic = (
    modelo_preprocessamento_logistic.transform(df_validacao_ml)
    .select("label", "features")
    .cache()
)

print("Treino:", df_treino_logistic.count())

print("Validação:", df_validacao_logistic.count())

## 8.2 Busca de Hiperparâmetros

São testadas poucas combinações de `regParam` e `elasticNetParam`.

O objetivo é verificar se a regularização melhora a capacidade de generalização sem realizar uma busca excessivamente ampla.

In [0]:
configs_logistic = [
    {
        "regParam": 0.0,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.001,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.1,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.5,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 1.0,
    },
]

In [0]:
resultados_logistic = []
modelos_logistic = {}

for i, config in enumerate(configs_logistic, start=1):
    nome_modelo = f"LR_{i:02d}"

    logistic = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxIter=100,
        regParam=config["regParam"],
        elasticNetParam=config["elasticNetParam"],
    )

    modelo = logistic.fit(df_treino_logistic)

    pred = modelo.transform(df_validacao_logistic)

    auc = evaluator_auc.evaluate(pred)

    metricas = calcular_metricas_classificacao(pred)

    resultados_logistic.append(
        (
            nome_modelo,
            config["regParam"],
            config["elasticNetParam"],
            metricas["accuracy"],
            auc,
            metricas["recall_0"],
            metricas["f1_0"],
        )
    )

    modelos_logistic[nome_modelo] = modelo

In [0]:
df_resultados_logistic = spark.createDataFrame(
    resultados_logistic,
    [
        "modelo",
        "regParam",
        "elasticNetParam",
        "accuracy",
        "roc_auc",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_resultados_logistic.orderBy(F.desc("roc_auc")))

## 8.3 Melhor configuração

In [0]:
melhor_logistic = df_resultados_logistic.orderBy(F.desc("roc_auc")).first()

print("Melhor configuração:")

print(f"Modelo: {melhor_logistic['modelo']}")

print(f"regParam: {melhor_logistic['regParam']}")

print(f"elasticNetParam: {melhor_logistic['elasticNetParam']}")

print(f"ROC-AUC: {melhor_logistic['roc_auc']:.4f}")

print(f"Recall classe 0: {melhor_logistic['recall_classe_0']:.4f}")

print(f"F1 classe 0: {melhor_logistic['f1_classe_0']:.4f}")

## 8.4 Resultado da Otimização

A melhor configuração foi obtida com `regParam = 0.1` e regularização L2, alcançando ROC-AUC de **0,6108**.

O resultado representa melhora em relação à Logistic Regression original da V2, cujo ROC-AUC foi de 0,6064. Entretanto, o ganho foi pequeno e acompanhado de redução no recall da classe `0`.

A Logistic Regression otimizada permanece como referência linear, mas ainda não supera a Random Forest V2, com ROC-AUC de 0,6130.

# 9. Otimização da Random Forest

A Random Forest apresentou o melhor resultado da V2, com ROC-AUC de **0,6130**.

Nesta etapa são avaliadas diferentes combinações de número de árvores, profundidade e regularização, mantendo as mesmas features e o mesmo conjunto de validação.

A configuração original também é incluída como referência.

## 9.1 Liberar os dados da Logistic

In [0]:
df_treino_logistic.unpersist()
df_validacao_logistic.unpersist()

## 9.2 Preprocessing para modelos de árvore

In [0]:
pipeline_preprocessamento_tree = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler_tree,
    ]
)

modelo_preprocessamento_tree = pipeline_preprocessamento_tree.fit(df_treino_ml)

In [0]:
df_treino_tree = (
    modelo_preprocessamento_tree.transform(df_treino_ml)
    .select("label", "features")
    .cache()
)

df_validacao_tree = (
    modelo_preprocessamento_tree.transform(df_validacao_ml)
    .select("label", "features")
    .cache()
)

print("Treino:", df_treino_tree.count())

print("Validação:", df_validacao_tree.count())

## 9.3 Busca de Hiperparâmetros

A busca é mantida reduzida para avaliar alterações relevantes sem ampliar excessivamente o custo computacional.

São testados número de árvores, profundidade, quantidade mínima de registros por nó e amostragem.

In [0]:
configs_rf = [
    {
        "numTrees": 50,
        "maxDepth": 8,
        "minInstancesPerNode": 1,
        "subsamplingRate": 1.0,
    },
    {
        "numTrees": 100,
        "maxDepth": 8,
        "minInstancesPerNode": 1,
        "subsamplingRate": 1.0,
    },
    {
        "numTrees": 100,
        "maxDepth": 10,
        "minInstancesPerNode": 1,
        "subsamplingRate": 1.0,
    },
    {
        "numTrees": 100,
        "maxDepth": 10,
        "minInstancesPerNode": 10,
        "subsamplingRate": 1.0,
    },
    {
        "numTrees": 100,
        "maxDepth": 10,
        "minInstancesPerNode": 30,
        "subsamplingRate": 1.0,
    },
    {
        "numTrees": 100,
        "maxDepth": 10,
        "minInstancesPerNode": 10,
        "subsamplingRate": 0.8,
    },
]

## 9.4 Treino

In [0]:
resultados_rf = []

melhor_auc_rf = 0.0
melhor_modelo_rf = None
melhor_config_rf = None

for i, config in enumerate(configs_rf, start=1):
    nome_modelo = f"RF_{i:02d}"

    random_forest = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=config["numTrees"],
        maxDepth=config["maxDepth"],
        minInstancesPerNode=config["minInstancesPerNode"],
        subsamplingRate=config["subsamplingRate"],
        seed=42,
    )

    modelo = random_forest.fit(df_treino_tree)

    pred = modelo.transform(df_validacao_tree)

    auc = evaluator_auc.evaluate(pred)

    metricas = calcular_metricas_classificacao(pred)

    resultados_rf.append(
        (
            nome_modelo,
            config["numTrees"],
            config["maxDepth"],
            config["minInstancesPerNode"],
            config["subsamplingRate"],
            metricas["accuracy"],
            auc,
            metricas["recall_0"],
            metricas["f1_0"],
        )
    )

    if auc > melhor_auc_rf:
        melhor_auc_rf = auc
        melhor_modelo_rf = modelo
        melhor_config_rf = config

    print(
        f"{nome_modelo} | "
        f"trees={config['numTrees']} | "
        f"depth={config['maxDepth']} | "
        f"minNode={config['minInstancesPerNode']} | "
        f"sample={config['subsamplingRate']} | "
        f"AUC={auc:.4f}"
    )

## 9.5 Resultados

In [0]:
df_resultados_rf = spark.createDataFrame(
    resultados_rf,
    [
        "modelo",
        "numTrees",
        "maxDepth",
        "minInstancesPerNode",
        "subsamplingRate",
        "accuracy",
        "roc_auc",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_resultados_rf.orderBy(F.desc("roc_auc")))

## 9.6 Melhor configuração

In [0]:
melhor_rf = df_resultados_rf.orderBy(F.desc("roc_auc")).first()

print("Melhor configuração:")
print(f"Modelo: {melhor_rf['modelo']}")
print(f"numTrees: {melhor_rf['numTrees']}")
print(f"maxDepth: {melhor_rf['maxDepth']}")
print(f"minInstancesPerNode: {melhor_rf['minInstancesPerNode']}")
print(f"subsamplingRate: {melhor_rf['subsamplingRate']}")
print(f"ROC-AUC: {melhor_rf['roc_auc']:.4f}")
print(f"Recall classe 0: {melhor_rf['recall_classe_0']:.4f}")
print(f"F1 classe 0: {melhor_rf['f1_classe_0']:.4f}")

## 9.7 Verificação Overfitting

In [0]:
pred_rf_treino = melhor_modelo_rf.transform(df_treino_tree)

auc_rf_treino = evaluator_auc.evaluate(pred_rf_treino)

gap_rf = auc_rf_treino - melhor_auc_rf

print(f"ROC-AUC treino: {auc_rf_treino:.4f}")

print(f"ROC-AUC validação: {melhor_auc_rf:.4f}")

print(f"Gap treino - validação: {gap_rf:.4f}")

## 9.8 Resultado da Otimização

A melhor configuração da Random Forest utilizou **100 árvores**, profundidade máxima de **10** e mínimo de **30 registros por nó**, alcançando ROC-AUC de **0,6164**.

O resultado representa melhora em relação à Random Forest original da V2, cujo ROC-AUC foi de 0,6130.

No conjunto de treino, o modelo apresentou ROC-AUC de **0,6695**, resultando em um gap de aproximadamente **0,053** em relação à validação. O resultado indica algum grau de overfitting, sem ganho suficiente para justificar aumento adicional da complexidade do modelo.

A Random Forest otimizada passa a ser a melhor referência de ROC-AUC até o momento.

# 10. Gradient Boosted Trees

O Gradient Boosted Trees é avaliado como alternativa aos modelos anteriores.

Diferentemente da Random Forest, o GBT constrói árvores sequencialmente, buscando corrigir os erros dos modelos anteriores. Essa abordagem pode capturar relações não lineares de forma mais eficiente.

Considerando o custo computacional e o prazo do projeto, será realizada uma busca reduzida de hiperparâmetros.

## 10.1 Configurações

In [0]:
configs_gbt = [
    {
        "maxIter": 30,
        "maxDepth": 3,
        "stepSize": 0.1,
        "minInstancesPerNode": 1,
    },
    {
        "maxIter": 50,
        "maxDepth": 3,
        "stepSize": 0.1,
        "minInstancesPerNode": 1,
    },
    {
        "maxIter": 30,
        "maxDepth": 5,
        "stepSize": 0.1,
        "minInstancesPerNode": 10,
    },
    {
        "maxIter": 50,
        "maxDepth": 5,
        "stepSize": 0.05,
        "minInstancesPerNode": 30,
    },
]

## 10.2 Treino

In [0]:
resultados_gbt = []

melhor_auc_gbt = 0.0
melhor_modelo_gbt = None
melhor_config_gbt = None

for i, config in enumerate(configs_gbt, start=1):
    nome_modelo = f"GBT_{i:02d}"

    gbt = GBTClassifier(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        maxIter=config["maxIter"],
        maxDepth=config["maxDepth"],
        stepSize=config["stepSize"],
        minInstancesPerNode=config["minInstancesPerNode"],
        seed=42,
    )

    modelo = gbt.fit(df_treino_tree)

    pred = modelo.transform(df_validacao_tree)

    auc = evaluator_auc.evaluate(pred)

    metricas = calcular_metricas_classificacao(pred)

    resultados_gbt.append(
        (
            nome_modelo,
            config["maxIter"],
            config["maxDepth"],
            config["stepSize"],
            config["minInstancesPerNode"],
            metricas["accuracy"],
            auc,
            metricas["recall_0"],
            metricas["f1_0"],
        )
    )

    if auc > melhor_auc_gbt:
        melhor_auc_gbt = auc
        melhor_modelo_gbt = modelo
        melhor_config_gbt = config

    print(
        f"{nome_modelo} | "
        f"iter={config['maxIter']} | "
        f"depth={config['maxDepth']} | "
        f"step={config['stepSize']} | "
        f"minNode={config['minInstancesPerNode']} | "
        f"AUC={auc:.4f}"
    )

## 10.3 Resultados

In [0]:
df_resultados_gbt = spark.createDataFrame(
    resultados_gbt,
    [
        "modelo",
        "maxIter",
        "maxDepth",
        "stepSize",
        "minInstancesPerNode",
        "accuracy",
        "roc_auc",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(
    df_resultados_gbt
    .orderBy(
        F.desc("roc_auc")
    )
)

## 10.4 Melhor Configiuração

In [0]:
melhor_gbt = df_resultados_gbt.orderBy(F.desc("roc_auc")).first()

print("Melhor configuração:")
print(f"Modelo: {melhor_gbt['modelo']}")
print(f"maxIter: {melhor_gbt['maxIter']}")
print(f"maxDepth: {melhor_gbt['maxDepth']}")
print(f"stepSize: {melhor_gbt['stepSize']}")
print(f"minInstancesPerNode: {melhor_gbt['minInstancesPerNode']}")
print(f"ROC-AUC: {melhor_gbt['roc_auc']:.4f}")
print(f"Recall classe 0: {melhor_gbt['recall_classe_0']:.4f}")
print(f"F1 classe 0: {melhor_gbt['f1_classe_0']:.4f}")

## 10.5 Overfitting do melhor GBT

In [0]:
pred_gbt_treino = melhor_modelo_gbt.transform(df_treino_tree)

auc_gbt_treino = evaluator_auc.evaluate(pred_gbt_treino)

gap_gbt = auc_gbt_treino - melhor_auc_gbt

print(f"ROC-AUC treino: {auc_gbt_treino:.4f}")

print(f"ROC-AUC validação: {melhor_auc_gbt:.4f}")

print(f"Gap treino - validação: {gap_gbt:.4f}")

## 10.6 Resultado do Gradient Boosted Trees

A melhor configuração do Gradient Boosted Trees utilizou **30 iterações**, profundidade máxima de **3** e `stepSize = 0.1`, alcançando ROC-AUC de **0,6135**.

O modelo apresentou recall de **35,15%** e F1-score de **0,4241** para a classe `0`, superando a Random Forest otimizada nessas métricas.

O gap entre treino e validação foi de aproximadamente **0,032**, inferior ao observado na Random Forest, indicando menor sinal de overfitting.

Apesar disso, a Random Forest permanece com o maior ROC-AUC da etapa de otimização, com **0,6164**.

# 11. Comparação dos Modelos Otimizados

Os melhores resultados de cada algoritmo são consolidados para apoiar a seleção do modelo final.

O ROC-AUC permanece como principal métrica de seleção, enquanto recall e F1-score da classe `0` são utilizados para avaliar a capacidade de identificar alunos não alfabetizados.

In [0]:
resultados_otimizados = [
    (
        "Logistic Regression",
        melhor_logistic["roc_auc"],
        melhor_logistic["accuracy"],
        melhor_logistic["recall_classe_0"],
        melhor_logistic["f1_classe_0"],
    ),
    (
        "Random Forest",
        melhor_rf["roc_auc"],
        melhor_rf["accuracy"],
        melhor_rf["recall_classe_0"],
        melhor_rf["f1_classe_0"],
    ),
    (
        "Gradient Boosted Trees",
        melhor_gbt["roc_auc"],
        melhor_gbt["accuracy"],
        melhor_gbt["recall_classe_0"],
        melhor_gbt["f1_classe_0"],
    ),
]

df_resultados_otimizados = spark.createDataFrame(
    resultados_otimizados,
    [
        "modelo",
        "roc_auc",
        "accuracy",
        "recall_classe_0",
        "f1_classe_0",
    ],
)

display(df_resultados_otimizados.orderBy(F.desc("roc_auc")))

# 12. Ajuste do Threshold da Random Forest

A Random Forest otimizada apresentou o maior ROC-AUC, porém ainda possui recall limitado para a classe de alunos não alfabetizados.

Como o threshold padrão de classificação é 0,5, serão avaliados diferentes pontos de corte utilizando a mesma probabilidade produzida pelo modelo.

O ROC-AUC não é alterado por esse ajuste; o objetivo é buscar melhor equilíbrio entre as métricas das classes.

In [0]:
pred_rf_threshold = melhor_modelo_rf.transform(df_validacao_tree).withColumn(
    "probabilidade_1", vector_to_array("probability")[1]
)

thresholds = [
    0.50,
    0.55,
    0.60,
    0.65,
]

resultados_threshold = []

for threshold in thresholds:
    pred_threshold = pred_rf_threshold.withColumn(
        "prediction",
        F.when(F.col("probabilidade_1") >= threshold, F.lit(1.0)).otherwise(F.lit(0.0)),
    )

    metricas = calcular_metricas_classificacao(pred_threshold)

    resultados_threshold.append(
        (
            threshold,
            metricas["accuracy"],
            metricas["precision_0"],
            metricas["recall_0"],
            metricas["f1_0"],
            metricas["precision_1"],
            metricas["recall_1"],
            metricas["f1_1"],
        )
    )

In [0]:
df_threshold = spark.createDataFrame(
    resultados_threshold,
    [
        "threshold",
        "accuracy",
        "precision_0",
        "recall_0",
        "f1_0",
        "precision_1",
        "recall_1",
        "f1_1",
    ],
)

display(df_threshold.orderBy("threshold"))

## 12.1 Seleção do Threshold

O threshold de **0,55** foi selecionado para a Random Forest otimizada.

Em comparação com o threshold padrão de 0,50, essa configuração reduziu pouco a acurácia global, mas aumentou de forma relevante a identificação dos alunos não alfabetizados.

Com `threshold = 0,55`, o modelo apresentou:

- accuracy de **59,66%**;
- precision da classe `0` de **51,94%**;
- recall da classe `0` de **46,66%**;
- F1-score da classe `0` de **0,4916**.

A escolha prioriza melhor equilíbrio na identificação da classe de maior interesse, sem alteração do ROC-AUC do modelo.

# 13. Seleção do Modelo Final

Após a otimização, a **Random Forest** foi selecionada como modelo final por apresentar o maior ROC-AUC na base de validação.

A configuração selecionada é:

- `numTrees = 100`;
- `maxDepth = 10`;
- `minInstancesPerNode = 30`;
- `subsamplingRate = 1.0`;
- `threshold = 0.55`.

O modelo alcançou ROC-AUC de **0,6164** na validação.

O threshold de 0,55 foi definido utilizando exclusivamente a base de validação de 2023, buscando melhorar a identificação dos alunos não alfabetizados sem perda relevante de acurácia global.

In [0]:
config_modelo_final = {
    "modelo": "Random Forest",
    "numTrees": 100,
    "maxDepth": 10,
    "minInstancesPerNode": 30,
    "subsamplingRate": 1.0,
    "threshold": 0.55,
    "roc_auc_validacao": melhor_auc_rf,
}

config_modelo_final